# Notebook 01: Transformer Architecture Fundamentals

## 📚 Historical Context

**Timeline**: June 2017 - "Attention is All You Need" paper published

**The Problem Before Transformers (Pre-2017)**:
- **RNNs/LSTMs**: Sequential processing → slow, can't parallelize
- **Long sequences**: Gradients vanish, model forgets early tokens
- **Translation**: Encoder compresses entire sentence into single vector → information bottleneck
- **Training time**: Weeks to train on large datasets

**The Breakthrough**:
- **Self-attention**: Every token attends to every other token directly
- **Parallelization**: All tokens processed simultaneously
- **No recurrence**: Fixed number of operations regardless of sequence length
- **Scalability**: Can scale to billions of parameters

**Impact**:
- 2018: BERT (Encoder-only) → revolutionized NLU
- 2018: GPT-1 (Decoder-only) → started the GPT series
- 2019: T5 (Encoder-Decoder) → unified text-to-text framework
- 2020+: GPT-3, ChatGPT, Llama → modern LLMs

## 🎯 What You'll Learn

1. **Self-Attention Mechanism** - The core innovation
2. **Multi-Head Attention** - Why multiple attention heads
3. **Positional Encoding** - How models know token order
4. **Feed-Forward Networks** - The other half of transformers
5. **Layer Normalization** - Stabilizing training
6. **Architecture Variants** - Encoder, Decoder, Encoder-Decoder

## 💡 Why This Matters

Understanding transformers is CRITICAL because:
- **All modern LLMs use transformers** (GPT, Llama, Mistral, etc.)
- **Fine-tuning decisions depend on architecture** (which layers to freeze/train)
- **Memory usage patterns** (attention is O(n²), feed-forward is O(n))
- **Debugging training issues** (attention collapse, gradient flow)

---

In [ ]:
# Import libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Set plot style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Part 1: Self-Attention Mechanism

### The Core Idea:
For each word, look at ALL other words and decide how relevant they are.

### Example:
Sentence: "The cat sat on the mat"

When processing "sat":
- High attention to "cat" (who sat?)
- High attention to "mat" (where?)
- Low attention to "the" (not very informative)

### Mathematical Formulation:

**Step 1: Create Query, Key, Value vectors**
```
For each token embedding x:
Query  (Q) = x @ W_Q    # "What am I looking for?"
Key    (K) = x @ W_K    # "What do I contain?"
Value  (V) = x @ W_V    # "What do I output if attended to?"
```

**Step 2: Compute attention scores**
```
Scores = Q @ K^T / sqrt(d_k)    # How similar is my query to each key?
```

**Step 3: Softmax to get attention weights**
```
Attention_Weights = softmax(Scores)    # Normalize to probabilities
```

**Step 4: Weighted sum of values**
```
Output = Attention_Weights @ V    # Weighted combination of values
```

### Why this works:
- **Query**: What information does this token need?
- **Key**: What information does this token provide?
- **Value**: The actual information to pass forward
- **Dot product**: Measures similarity between query and keys
- **Scaling by sqrt(d_k)**: Prevents softmax saturation for large dimensions

In [ ]:
class SelfAttention(nn.Module):
    """
    Self-Attention mechanism from "Attention is All You Need"
    
    This is the CORE of transformers. Understanding this is essential.
    """
    def __init__(self, embed_dim, head_dim):
        """
        Args:
            embed_dim: Dimension of input embeddings (e.g., 768 for BERT-base)
            head_dim: Dimension of each attention head (typically embed_dim // num_heads)
        """
        super().__init__()
        self.embed_dim = embed_dim
        self.head_dim = head_dim
        
        # Linear projections for Q, K, V
        # These are learnable transformations
        self.query_proj = nn.Linear(embed_dim, head_dim)
        self.key_proj = nn.Linear(embed_dim, head_dim)
        self.value_proj = nn.Linear(embed_dim, head_dim)
        
        # Scaling factor to prevent softmax saturation
        # Without this, dot products can become very large, pushing softmax to extremes
        self.scale = math.sqrt(head_dim)
    
    def forward(self, x, mask=None, return_attention=False):
        """
        Args:
            x: Input tensor [batch_size, seq_len, embed_dim]
            mask: Optional attention mask [batch_size, seq_len, seq_len]
                  Used for padding and causal attention
            return_attention: If True, also return attention weights for visualization
        
        Returns:
            output: Attention output [batch_size, seq_len, head_dim]
            attention_weights (optional): [batch_size, seq_len, seq_len]
        """
        batch_size, seq_len, embed_dim = x.shape
        
        # Step 1: Project to Q, K, V
        # Each token gets its own query, key, and value vectors
        Q = self.query_proj(x)  # [batch_size, seq_len, head_dim]
        K = self.key_proj(x)    # [batch_size, seq_len, head_dim]
        V = self.value_proj(x)  # [batch_size, seq_len, head_dim]
        
        # Step 2: Compute attention scores
        # Q @ K^T gives similarity between each pair of tokens
        # Shape: [batch_size, seq_len, seq_len]
        # scores[i, j] = how much token i attends to token j
        scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale
        
        # Step 3: Apply mask if provided
        if mask is not None:
            # Replace masked positions with -inf so softmax gives them 0 weight
            # This is used for:
            # 1. Padding tokens (don't attend to padding)
            # 2. Causal masking (future tokens can't attend to past in decoder)
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        # Step 4: Softmax to get attention weights
        # Converts scores to probabilities (sum to 1 for each token)
        attention_weights = F.softmax(scores, dim=-1)
        
        # Step 5: Weighted sum of values
        # Each token's output is a weighted combination of all value vectors
        output = torch.matmul(attention_weights, V)  # [batch_size, seq_len, head_dim]
        
        if return_attention:
            return output, attention_weights
        return output

# Test the self-attention mechanism
print("=" * 60)
print("TESTING SELF-ATTENTION")
print("=" * 60)

# Create a simple example
batch_size = 2
seq_len = 5
embed_dim = 128
head_dim = 64

# Random input (in practice, this would be word embeddings)
x = torch.randn(batch_size, seq_len, embed_dim)

# Create attention module
attention = SelfAttention(embed_dim, head_dim)

# Forward pass
output, attention_weights = attention(x, return_attention=True)

print(f"Input shape:             {x.shape}")
print(f"Output shape:            {output.shape}")
print(f"Attention weights shape: {attention_weights.shape}")
print(f"\nAttention weights sum (should be 1.0 per row):")
print(attention_weights[0].sum(dim=-1))  # Each row sums to 1
print("=" * 60)

## Part 2: Visualizing Attention

Let's visualize attention patterns on a real sentence.

In [ ]:
def visualize_attention(attention_weights, tokens, title="Attention Pattern"):
    """
    Visualize attention weights as a heatmap.
    
    Args:
        attention_weights: [seq_len, seq_len] attention matrix
        tokens: List of token strings
        title: Plot title
    """
    plt.figure(figsize=(10, 8))
    
    # Create heatmap
    sns.heatmap(
        attention_weights.cpu().numpy(),
        xticklabels=tokens,
        yticklabels=tokens,
        cmap='YlOrRd',
        annot=True,
        fmt='.2f',
        square=True,
        cbar_kws={'label': 'Attention Weight'}
    )
    
    plt.xlabel('Keys (attending TO)')
    plt.ylabel('Queries (attending FROM)')
    plt.title(title)
    plt.tight_layout()
    plt.show()

# Simple example with meaningful tokens
tokens = ["The", "cat", "sat", "on", "mat"]
seq_len = len(tokens)

# Create dummy embeddings (in practice, these come from an embedding layer)
# For demonstration, we'll create embeddings that make semantic sense
embed_dim = 64
embeddings = torch.randn(1, seq_len, embed_dim)

# Apply attention
attention = SelfAttention(embed_dim, embed_dim)
_, attn_weights = attention(embeddings, return_attention=True)

# Visualize
visualize_attention(
    attn_weights[0],  # First batch
    tokens,
    "Self-Attention Pattern"
)

print("\nInterpretation:")
print("- Diagonal: Tokens attend to themselves")
print("- Bright cells: Strong attention between tokens")
print("- Dark cells: Weak attention between tokens")
print("\nIn a trained model, you'd see:")
print("- 'sat' attending strongly to 'cat' (subject-verb)")
print("- 'sat' attending strongly to 'mat' (verb-location)")
print("- 'The' attending weakly (less semantic content)")

## Part 3: Multi-Head Attention

### Why Multiple Heads?

**Single head limitation**: Can only learn ONE type of relationship

**Multi-head advantage**: Different heads learn different relationships:
- Head 1: Syntactic relationships (subject-verb, verb-object)
- Head 2: Semantic relationships (synonyms, antonyms)
- Head 3: Positional relationships (nearby words)
- Head 4: Long-range dependencies

### Implementation:
```
1. Split embed_dim into num_heads
   Example: 768 dim → 12 heads × 64 dim per head

2. Run self-attention on each head independently

3. Concatenate all head outputs

4. Project back to embed_dim
```

### Key Insight:
More heads = more diverse attention patterns = richer representations

**Typical configurations**:
- BERT-base: 12 heads, 768 dim → 64 dim per head
- GPT-2: 12 heads, 768 dim → 64 dim per head
- Llama-7B: 32 heads, 4096 dim → 128 dim per head

In [ ]:
class MultiHeadAttention(nn.Module):
    """
    Multi-Head Attention mechanism.
    
    This allows the model to jointly attend to information from different
    representation subspaces at different positions.
    """
    def __init__(self, embed_dim, num_heads, dropout=0.1):
        """
        Args:
            embed_dim: Model dimension (must be divisible by num_heads)
            num_heads: Number of attention heads
            dropout: Dropout probability for attention weights
        """
        super().__init__()
        assert embed_dim % num_heads == 0, "embed_dim must be divisible by num_heads"
        
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads  # Dimension per head
        self.scale = math.sqrt(self.head_dim)
        
        # Single projection matrices for all heads (more efficient)
        # We'll split them into heads later
        self.qkv_proj = nn.Linear(embed_dim, 3 * embed_dim)  # Q, K, V in one go
        
        # Output projection
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        
        # Dropout for regularization
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None, return_attention=False):
        """
        Args:
            x: [batch_size, seq_len, embed_dim]
            mask: Optional [batch_size, 1, seq_len, seq_len]
        
        Returns:
            output: [batch_size, seq_len, embed_dim]
        """
        batch_size, seq_len, embed_dim = x.shape
        
        # Step 1: Project to Q, K, V
        # qkv shape: [batch_size, seq_len, 3 * embed_dim]
        qkv = self.qkv_proj(x)
        
        # Step 2: Split into Q, K, V
        # Each has shape: [batch_size, seq_len, embed_dim]
        Q, K, V = qkv.chunk(3, dim=-1)
        
        # Step 3: Reshape to separate heads
        # [batch_size, seq_len, num_heads, head_dim]
        Q = Q.view(batch_size, seq_len, self.num_heads, self.head_dim)
        K = K.view(batch_size, seq_len, self.num_heads, self.head_dim)
        V = V.view(batch_size, seq_len, self.num_heads, self.head_dim)
        
        # Step 4: Transpose to [batch_size, num_heads, seq_len, head_dim]
        # This allows us to process all heads in parallel
        Q = Q.transpose(1, 2)
        K = K.transpose(1, 2)
        V = V.transpose(1, 2)
        
        # Step 5: Compute attention scores for all heads
        # [batch_size, num_heads, seq_len, seq_len]
        scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale
        
        # Step 6: Apply mask if provided
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        # Step 7: Softmax
        attention_weights = F.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)
        
        # Step 8: Apply attention to values
        # [batch_size, num_heads, seq_len, head_dim]
        attn_output = torch.matmul(attention_weights, V)
        
        # Step 9: Concatenate heads
        # Transpose back to [batch_size, seq_len, num_heads, head_dim]
        attn_output = attn_output.transpose(1, 2).contiguous()
        
        # Reshape to [batch_size, seq_len, embed_dim]
        attn_output = attn_output.view(batch_size, seq_len, embed_dim)
        
        # Step 10: Final output projection
        output = self.out_proj(attn_output)
        
        if return_attention:
            return output, attention_weights
        return output

# Test multi-head attention
print("=" * 60)
print("TESTING MULTI-HEAD ATTENTION")
print("=" * 60)

batch_size = 2
seq_len = 10
embed_dim = 512
num_heads = 8

x = torch.randn(batch_size, seq_len, embed_dim)
mha = MultiHeadAttention(embed_dim, num_heads)

output, attn_weights = mha(x, return_attention=True)

print(f"Input shape:       {x.shape}")
print(f"Output shape:      {output.shape}")
print(f"Attention weights: {attn_weights.shape}")
print(f"                   [batch, num_heads, seq_len, seq_len]")
print(f"\nNumber of heads:   {num_heads}")
print(f"Dim per head:      {embed_dim // num_heads}")

# Count parameters
total_params = sum(p.numel() for p in mha.parameters())
print(f"\nTotal parameters:  {total_params:,}")
print("=" * 60)

## Part 4: Positional Encoding

### The Problem:
Self-attention is **permutation invariant** - it doesn't care about token order!

```python
"The cat sat on the mat" 
"mat the on sat cat The"
```
Without positional information, these would be treated identically!

### The Solution:
Add position information to embeddings.

### Two Approaches:

**1. Sinusoidal Positional Encoding (Original Transformer, 2017)**
- Fixed, not learned
- Based on sine/cosine waves of different frequencies
- Can generalize to longer sequences than seen during training
- Formula:
```
PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
```

**2. Learned Positional Embeddings (BERT, GPT, 2018+)**
- Learned during training
- More flexible, can adapt to task
- Limited to max sequence length seen during training
- Simply a lookup table: position → embedding vector

**Modern Approach: Rotary Position Embedding (RoPE, 2021)**
- Used in Llama, GPT-NeoX, PaLM
- Applies rotation to Q and K based on position
- Better for long sequences
- We'll cover this in Stage 3 (long context training)

In [ ]:
class SinusoidalPositionalEncoding(nn.Module):
    """
    Sinusoidal positional encoding from "Attention is All You Need".
    
    Uses sine and cosine functions of different frequencies to encode position.
    """
    def __init__(self, embed_dim, max_len=5000, dropout=0.1):
        """
        Args:
            embed_dim: Dimension of embeddings
            max_len: Maximum sequence length to precompute
            dropout: Dropout probability
        """
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        
        # Create positional encoding matrix
        # Shape: [max_len, embed_dim]
        pe = torch.zeros(max_len, embed_dim)
        
        # Position indices: [0, 1, 2, ..., max_len-1]
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        
        # Dimension indices: [0, 2, 4, ..., embed_dim-2]
        # We use even indices for sine, odd indices for cosine
        div_term = torch.exp(
            torch.arange(0, embed_dim, 2).float() * (-math.log(10000.0) / embed_dim)
        )
        
        # Apply sine to even indices
        pe[:, 0::2] = torch.sin(position * div_term)
        
        # Apply cosine to odd indices
        pe[:, 1::2] = torch.cos(position * div_term)
        
        # Add batch dimension: [1, max_len, embed_dim]
        pe = pe.unsqueeze(0)
        
        # Register as buffer (not a parameter, but should be saved with model)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        """
        Args:
            x: [batch_size, seq_len, embed_dim]
        
        Returns:
            x + positional encoding: [batch_size, seq_len, embed_dim]
        """
        # Add positional encoding to input
        # self.pe[:, :x.size(1)] extracts the first seq_len positions
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

class LearnedPositionalEmbedding(nn.Module):
    """
    Learned positional embeddings (BERT, GPT style).
    
    Simply a lookup table that's learned during training.
    """
    def __init__(self, max_len, embed_dim, dropout=0.1):
        """
        Args:
            max_len: Maximum sequence length
            embed_dim: Embedding dimension
            dropout: Dropout probability
        """
        super().__init__()
        # Embedding table: position index → embedding vector
        self.pos_embedding = nn.Embedding(max_len, embed_dim)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        """
        Args:
            x: [batch_size, seq_len, embed_dim]
        
        Returns:
            x + positional embedding: [batch_size, seq_len, embed_dim]
        """
        batch_size, seq_len, embed_dim = x.shape
        
        # Create position indices: [0, 1, 2, ..., seq_len-1]
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0)
        
        # Look up embeddings for these positions
        pos_emb = self.pos_embedding(positions)  # [1, seq_len, embed_dim]
        
        # Add to input
        x = x + pos_emb
        return self.dropout(x)

# Visualize positional encodings
print("=" * 60)
print("POSITIONAL ENCODING COMPARISON")
print("=" * 60)

embed_dim = 128
max_len = 100

# Create both types
sinusoidal_pe = SinusoidalPositionalEncoding(embed_dim, max_len)
learned_pe = LearnedPositionalEmbedding(max_len, embed_dim)

# Test input
x = torch.randn(1, 50, embed_dim)

# Apply both
x_sin = sinusoidal_pe(x)
x_learned = learned_pe(x)

print(f"Input shape:           {x.shape}")
print(f"After sinusoidal PE:   {x_sin.shape}")
print(f"After learned PE:      {x_learned.shape}")

# Visualize sinusoidal PE
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Sinusoidal PE heatmap
pe_matrix = sinusoidal_pe.pe[0, :50, :64].cpu().numpy()  # First 50 positions, 64 dims
im1 = axes[0].imshow(pe_matrix.T, aspect='auto', cmap='RdBu', vmin=-1, vmax=1)
axes[0].set_xlabel('Position')
axes[0].set_ylabel('Embedding Dimension')
axes[0].set_title('Sinusoidal Positional Encoding')
plt.colorbar(im1, ax=axes[0])

# Plot 2: Different frequencies
positions = np.arange(50)
axes[1].plot(positions, pe_matrix[:, 0], label='Dim 0 (high freq)', alpha=0.7)
axes[1].plot(positions, pe_matrix[:, 16], label='Dim 16', alpha=0.7)
axes[1].plot(positions, pe_matrix[:, 32], label='Dim 32', alpha=0.7)
axes[1].plot(positions, pe_matrix[:, 48], label='Dim 48 (low freq)', alpha=0.7)
axes[1].set_xlabel('Position')
axes[1].set_ylabel('Encoding Value')
axes[1].set_title('Different Frequencies in Sinusoidal PE')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKey Observations:")
print("1. Each dimension uses a different frequency")
print("2. Lower dimensions = higher frequencies (change quickly)")
print("3. Higher dimensions = lower frequencies (change slowly)")
print("4. This allows the model to attend to relative positions")
print("=" * 60)

## Part 5: Feed-Forward Network

### Often Overlooked, But Critical:

**What it does**:
- Takes attention output
- Applies two linear transformations with ReLU in between
- Adds non-linearity and depth

**Architecture**:
```
FFN(x) = ReLU(x @ W1 + b1) @ W2 + b2

Where:
W1: [embed_dim, ff_dim]    # Typically ff_dim = 4 * embed_dim
W2: [ff_dim, embed_dim]    # Project back
```

**Why 4x expansion?**
- Original paper found 4x works well empirically
- More parameters → more expressiveness
- Most of transformer's parameters are here!

**Parameter Count**:
```
Attention: ~4 * embed_dim^2
FFN:       ~8 * embed_dim^2  (with 4x expansion)
```

**Modern Variants**:
- **GELU activation** (GPT, BERT): Smoother than ReLU
- **SwiGLU** (Llama, PaLM): Even better performance
- **Gated Linear Units**: Two parallel projections

In [ ]:
class FeedForward(nn.Module):
    """
    Position-wise Feed-Forward Network.
    
    Applied independently to each position.
    """
    def __init__(self, embed_dim, ff_dim, dropout=0.1, activation='relu'):
        """
        Args:
            embed_dim: Model dimension
            ff_dim: Hidden dimension (typically 4 * embed_dim)
            dropout: Dropout probability
            activation: 'relu', 'gelu', or 'swiglu'
        """
        super().__init__()
        
        if activation == 'swiglu':
            # SwiGLU uses two parallel linear layers
            # One for the gate, one for the value
            self.w1 = nn.Linear(embed_dim, ff_dim)
            self.w2 = nn.Linear(embed_dim, ff_dim)
            self.w3 = nn.Linear(ff_dim, embed_dim)
        else:
            # Standard FFN
            self.linear1 = nn.Linear(embed_dim, ff_dim)
            self.linear2 = nn.Linear(ff_dim, embed_dim)
        
        self.dropout = nn.Dropout(dropout)
        self.activation = activation
        
        # Activation function
        if activation == 'relu':
            self.act = F.relu
        elif activation == 'gelu':
            self.act = F.gelu
    
    def forward(self, x):
        """
        Args:
            x: [batch_size, seq_len, embed_dim]
        
        Returns:
            output: [batch_size, seq_len, embed_dim]
        """
        if self.activation == 'swiglu':
            # SwiGLU: SiLU(x @ W1) * (x @ W2) @ W3
            # SiLU = Sigmoid Linear Unit = x * sigmoid(x)
            return self.w3(F.silu(self.w1(x)) * self.w2(x))
        else:
            # Standard: activation(x @ W1) @ W2
            x = self.linear1(x)
            x = self.act(x)
            x = self.dropout(x)
            x = self.linear2(x)
            return x

# Test different FFN variants
print("=" * 60)
print("TESTING FEED-FORWARD NETWORKS")
print("=" * 60)

batch_size = 2
seq_len = 10
embed_dim = 512
ff_dim = 2048  # 4x expansion

x = torch.randn(batch_size, seq_len, embed_dim)

# Test each variant
for activation in ['relu', 'gelu', 'swiglu']:
    ffn = FeedForward(embed_dim, ff_dim, activation=activation)
    output = ffn(x)
    
    params = sum(p.numel() for p in ffn.parameters())
    
    print(f"\n{activation.upper()}:")
    print(f"  Input shape:  {x.shape}")
    print(f"  Output shape: {output.shape}")
    print(f"  Parameters:   {params:,}")

print("\n" + "=" * 60)
print("ACTIVATION FUNCTION COMPARISON")
print("=" * 60)

# Visualize activation functions
x_range = torch.linspace(-3, 3, 100)

plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.plot(x_range, F.relu(x_range), label='ReLU', linewidth=2)
plt.grid(True, alpha=0.3)
plt.xlabel('x')
plt.ylabel('ReLU(x)')
plt.title('ReLU Activation')
plt.legend()

plt.subplot(1, 3, 2)
plt.plot(x_range, F.gelu(x_range), label='GELU', color='orange', linewidth=2)
plt.grid(True, alpha=0.3)
plt.xlabel('x')
plt.ylabel('GELU(x)')
plt.title('GELU Activation')
plt.legend()

plt.subplot(1, 3, 3)
plt.plot(x_range, F.silu(x_range), label='SiLU (SwiGLU)', color='green', linewidth=2)
plt.grid(True, alpha=0.3)
plt.xlabel('x')
plt.ylabel('SiLU(x)')
plt.title('SiLU Activation')
plt.legend()

plt.tight_layout()
plt.show()

print("\nKey Differences:")
print("ReLU:   Sharp cutoff at 0, simple, fast")
print("GELU:   Smooth, probabilistic, used in BERT/GPT-2/GPT-3")
print("SwiGLU: Best performance, used in Llama/PaLM, but more parameters")
print("=" * 60)

## Part 6: Complete Transformer Block

Now let's put it all together into a complete transformer layer.

### Architecture:
```
Input
  ↓
Layer Norm
  ↓
Multi-Head Attention  ←─── Residual Connection
  ↓
Layer Norm
  ↓
Feed-Forward Network  ←─── Residual Connection
  ↓
Output
```

### Key Design Choices:

**1. Residual Connections** (from ResNet)
- Allow gradients to flow directly through network
- Prevent vanishing gradients in deep networks
- `output = layer(x) + x`

**2. Layer Normalization**
- Stabilizes training
- Two positions: Pre-LN (modern) vs Post-LN (original)
- Pre-LN: More stable, easier to train deep networks

**3. Dropout**
- Regularization to prevent overfitting
- Applied after attention and FFN
- Typical values: 0.1-0.3

In [ ]:
class TransformerBlock(nn.Module):
    """
    Complete Transformer block.
    
    This is the fundamental building block of all transformer models.
    Stack multiple blocks to create deep transformers.
    """
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1, pre_norm=True):
        """
        Args:
            embed_dim: Model dimension
            num_heads: Number of attention heads
            ff_dim: Feed-forward hidden dimension
            dropout: Dropout probability
            pre_norm: If True, use Pre-LN (modern). If False, use Post-LN (original)
        """
        super().__init__()
        self.pre_norm = pre_norm
        
        # Multi-head attention
        self.attention = MultiHeadAttention(embed_dim, num_heads, dropout)
        
        # Feed-forward network
        self.ffn = FeedForward(embed_dim, ff_dim, dropout)
        
        # Layer normalization
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        
        # Dropout
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        """
        Args:
            x: [batch_size, seq_len, embed_dim]
            mask: Optional attention mask
        
        Returns:
            output: [batch_size, seq_len, embed_dim]
        """
        if self.pre_norm:
            # Pre-LN: Normalize before each sub-layer
            # This is more stable and used in modern transformers
            
            # Attention block
            attn_output = self.attention(self.norm1(x), mask)
            x = x + self.dropout(attn_output)  # Residual connection
            
            # FFN block
            ffn_output = self.ffn(self.norm2(x))
            x = x + self.dropout(ffn_output)  # Residual connection
        else:
            # Post-LN: Normalize after each sub-layer (original paper)
            # Less stable for very deep networks
            
            # Attention block
            attn_output = self.attention(x, mask)
            x = self.norm1(x + self.dropout(attn_output))
            
            # FFN block
            ffn_output = self.ffn(x)
            x = self.norm2(x + self.dropout(ffn_output))
        
        return x

# Test transformer block
print("=" * 60)
print("TESTING TRANSFORMER BLOCK")
print("=" * 60)

batch_size = 2
seq_len = 10
embed_dim = 512
num_heads = 8
ff_dim = 2048

x = torch.randn(batch_size, seq_len, embed_dim)

# Test both Pre-LN and Post-LN
for pre_norm in [True, False]:
    block = TransformerBlock(embed_dim, num_heads, ff_dim, pre_norm=pre_norm)
    output = block(x)
    
    params = sum(p.numel() for p in block.parameters())
    
    norm_type = "Pre-LN (modern)" if pre_norm else "Post-LN (original)"
    print(f"\n{norm_type}:")
    print(f"  Input shape:  {x.shape}")
    print(f"  Output shape: {output.shape}")
    print(f"  Parameters:   {params:,}")
    
    # Break down parameters
    attn_params = sum(p.numel() for p in block.attention.parameters())
    ffn_params = sum(p.numel() for p in block.ffn.parameters())
    norm_params = sum(p.numel() for p in block.norm1.parameters()) + \
                  sum(p.numel() for p in block.norm2.parameters())
    
    print(f"\n  Parameter breakdown:")
    print(f"    Attention:    {attn_params:,} ({attn_params/params*100:.1f}%)")
    print(f"    Feed-Forward: {ffn_params:,} ({ffn_params/params*100:.1f}%)")
    print(f"    Layer Norm:   {norm_params:,} ({norm_params/params*100:.1f}%)")

print("\n" + "=" * 60)

## Part 7: Architecture Variants

### Three Main Types:

**1. Encoder-only (BERT, RoBERTa, DeBERTa)**
- Bidirectional attention (can see all tokens)
- Best for: Classification, NER, question answering
- Example use: "Is this email spam?"

**2. Decoder-only (GPT, Llama, Mistral)**
- Causal attention (can only see past tokens)
- Best for: Text generation, autocompletion
- Example use: "Continue this story..."

**3. Encoder-Decoder (T5, BART, mT5)**
- Encoder: Bidirectional
- Decoder: Causal + Cross-attention to encoder
- Best for: Translation, summarization
- Example use: "Translate English to French"

### Modern Trend:
**Decoder-only dominates (2023+)**
- Simpler architecture
- Better scaling properties
- Can do everything with prompting
- All recent LLMs are decoder-only (GPT-4, Llama, Claude)

In [ ]:
def create_causal_mask(seq_len, device='cpu'):
    """
    Create causal (autoregressive) mask for decoder.
    
    Prevents attending to future tokens.
    
    Returns:
        mask: [seq_len, seq_len] lower triangular matrix
    """
    # Create lower triangular matrix of 1s
    # Position i can attend to positions 0 to i (inclusive)
    mask = torch.tril(torch.ones(seq_len, seq_len, device=device))
    return mask.unsqueeze(0).unsqueeze(0)  # [1, 1, seq_len, seq_len]

# Visualize masks
print("=" * 60)
print("ATTENTION MASKS")
print("=" * 60)

seq_len = 8

# Bidirectional mask (encoder)
bidirectional_mask = torch.ones(seq_len, seq_len)

# Causal mask (decoder)
causal_mask = create_causal_mask(seq_len)[0, 0]

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bidirectional
axes[0].imshow(bidirectional_mask, cmap='RdYlGn', vmin=0, vmax=1)
axes[0].set_title('Bidirectional Attention (Encoder)\nAll tokens can attend to all tokens')
axes[0].set_xlabel('Key Position')
axes[0].set_ylabel('Query Position')
for i in range(seq_len):
    for j in range(seq_len):
        axes[0].text(j, i, '1', ha='center', va='center', color='black')

# Causal
axes[1].imshow(causal_mask, cmap='RdYlGn', vmin=0, vmax=1)
axes[1].set_title('Causal Attention (Decoder)\nCan only attend to past tokens')
axes[1].set_xlabel('Key Position')
axes[1].set_ylabel('Query Position')
for i in range(seq_len):
    for j in range(seq_len):
        value = '1' if j <= i else '0'
        color = 'black' if j <= i else 'white'
        axes[1].text(j, i, value, ha='center', va='center', color=color)

plt.tight_layout()
plt.show()

print("\nKey Differences:")
print("\nEncoder (Bidirectional):")
print("  - Token 3 can see: tokens 0, 1, 2, 3, 4, 5, 6, 7")
print("  - Use case: Understand entire sentence context")
print("  - Example: BERT for classification")

print("\nDecoder (Causal):")
print("  - Token 3 can see: tokens 0, 1, 2, 3 only")
print("  - Use case: Generate text left-to-right")
print("  - Example: GPT for completion")

print("\n" + "=" * 60)

## 📊 Summary

### What We Learned:

1. **Self-Attention**
   - Q, K, V mechanism
   - Scaled dot-product attention
   - O(n²) complexity

2. **Multi-Head Attention**
   - Learn different relationship types
   - Parallel heads increase expressiveness
   - Most modern models: 12-32 heads

3. **Positional Encoding**
   - Attention is permutation invariant
   - Sinusoidal (fixed) vs Learned
   - Modern: RoPE (rotary position embedding)

4. **Feed-Forward Network**
   - 2/3 of transformer parameters
   - 4x hidden dimension expansion
   - ReLU → GELU → SwiGLU evolution

5. **Transformer Block**
   - Attention + FFN + Residuals + LayerNorm
   - Pre-LN vs Post-LN
   - Stack 12-96 layers for full model

6. **Architecture Variants**
   - Encoder-only: BERT (classification)
   - Decoder-only: GPT (generation) ← Most popular now
   - Encoder-Decoder: T5 (translation)

### Key Takeaways for Fine-Tuning:

**Memory Usage**:
```
Attention:     O(n²) in sequence length
FFN:           O(n) in sequence length
Longer sequences → dramatically more memory
```

**Parameter Distribution**:
```
Attention:     ~30-40% of parameters
FFN:           ~60-70% of parameters
Layer Norm:    <1% of parameters
```

**Fine-Tuning Strategies** (we'll explore in Stage 2):
- Freeze attention layers, train FFN layers
- LoRA on attention Q, V projections
- Full fine-tuning of last N layers only

### Model Size Examples:

| Model | Layers | Hidden | Heads | Params | Architecture |
|-------|--------|---------|-------|--------|-------------|
| BERT-base | 12 | 768 | 12 | 110M | Encoder |
| GPT-2 | 12 | 768 | 12 | 117M | Decoder |
| GPT-2 Large | 36 | 1280 | 20 | 774M | Decoder |
| Llama-7B | 32 | 4096 | 32 | 7B | Decoder |
| Llama-70B | 80 | 8192 | 64 | 70B | Decoder |

### Next Notebook:
**02_tokenization_deep_dive.ipynb**

We'll explore:
- How text becomes numbers
- BPE vs WordPiece vs SentencePiece
- Vocabulary size trade-offs
- Special tokens and their importance
- Common tokenization bugs

---

**Historical Note**: The original transformer (2017) had 6 encoder + 6 decoder layers. Modern models pushed to 96+ layers (GPT-3), then found that decoder-only with 32-80 layers works best (Llama). The trend is: deeper, decoder-only, better activation functions.